In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from tqdm import tqdm

from dataset_definition import UnifiedSeismicDataset
from model_definition import SeismicMagnitudePredictor

MODEL_PATH = "path/to/best_model.pth"
DATA_DIR   = "/home/aidar/study/senior_thesis/data/INSTANCE/clipped_amplitude"

BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


with open(os.path.join(DATA_DIR, "info.json")) as f:
    info = json.load(f)

n_samples = info["events"]["n_written"]

dataset = UnifiedSeismicDataset(
    memmap_path=os.path.join(DATA_DIR, "waveforms_events.bin"),
    csv_path=os.path.join(DATA_DIR, "metadata_events.csv"),
    n_samples=n_samples,
    phase="test",
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

model = SeismicMagnitudePredictor().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# manually extract backbone forward
def extract_embedding(model, x):
    x = model.spectrogram(x)
    x = torch.log1p(x)

    feats = model.cnn(x)
    lstm_input = feats.permute(0, 2, 1)

    lstm_out, _ = model.lstm(lstm_input)

    return lstm_out[:, -1, :]  # (batch, 64)

all_embeddings = []
all_magnitudes = []

print("Extracting embeddings...")

with torch.inference_mode():
    for inputs, targets in tqdm(loader):
        inputs = inputs.to(DEVICE, non_blocking=True)

        emb = extract_embedding(model, inputs)

        all_embeddings.append(emb.cpu().numpy())
        all_magnitudes.append(targets.numpy())

X = np.concatenate(all_embeddings, axis=0)   # (N, 64)
y = np.concatenate(all_magnitudes, axis=0)   # (N, 1)
y = y.squeeze()

print(f"Embeddings shape: {X.shape}")

pca = PCA(n_components=2) # just 2 components, so far it would be better for interpretability
X_pca = pca.fit_transform(X)

print(f"Explained variance: {pca.explained_variance_ratio_}")


def mag_bin(m):
    if m < 3.0:
        return 0  # low
    elif m < 5.0:
        return 1  # mid
    return 2      # high

bins = np.array([mag_bin(m) for m in y])

plt.figure(figsize=(12, 8), dpi=150)

colors = ["blue", "orange", "red"]
labels = ["Low [1–3)", "Mid [3–5)", "High [5+]"]

for i in range(3):
    idx = bins == i
    plt.scatter(
        X_pca[idx, 0],
        X_pca[idx, 1],
        s=10,
        alpha=0.5,
        label=labels[i]
    )

plt.title("PCA of learned embeddings (CNN + LSTM)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("embedding_pca.png")
plt.show()